In [1]:
setwd("C:\\Local_Files\\SCTLD_samples\\Sample_Data")
pan_data <- read.csv("PAN-BDT_ColonyData.csv")

# loading libraries
library(dplyr)
library(tidyr)
library(stringr)
library(knitr)

#reorganizing data into long format
long_data <- pan_data %>%
  filter( Transect_num %in% c("1", "2", "3", "4"), ) %>% #only keep transects 1-4
  pivot_longer(
    cols = c("X092022_Condition", "X102023_Condition", "X072024_Condition"),
    names_to = "time_point",          
    values_to = "health_status") %>%  
  filter(!is.na(health_status) & health_status != "") %>%      
  mutate(
    Species = str_trim(str_to_upper(Species)),                   #clean spaces/caps- species codes
    health_status = str_trim(str_to_lower(health_status)),       #clean spaces/ caps - health status
    time_date = case_when(
      time_point == "X092022_Condition" ~ as.Date("09-01-2022"),
      time_point == "X102023_Condition" ~ as.Date("10-01-2023"),
      time_point == "X072024_Condition" ~ as.Date("07-01-2024"),
      TRUE ~ as.Date(NA)),
    Species = case_when(Species %in% c("OANN", "OFAV") ~ "ORBI", TRUE ~ Species )) 

Warning message:
"package 'dplyr' was built under R version 4.4.3"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'tidyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
Warning message:
"package 'knitr' was built under R version 4.4.3"


In [4]:
#count non-healthy coral conditions per species
non_healthy_counts_species <- long_data %>%
  filter(health_status != "healthy") %>%               
  count(Species, sort = TRUE) %>%                     
  rename(non_healthy = n)                             

#total species counts 
total_counts_species <- long_data %>%
  count(Species, sort = TRUE) %>%                     
  rename(total = n)                                    

#calculating percent unhealthy per species
coral_summary_species <- total_counts_species %>%
  left_join(non_healthy_counts_species, by = "Species") %>%
  mutate(
    non_healthy = replace_na(non_healthy, 0),    
#some species may have no non-healthy cases so they’re missing from the non_healthy_counts table; 
#replaces those NA with 0, so the math doesn’t break; correctly reflects that no unhealthy corals were observed.
    healthy = total - non_healthy,                # creating healthy column 
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(desc(percent_unhealthy))   

kable(coral_summary_species, format = "markdown")        



|Species | total| non_healthy| healthy| percent_unhealthy|
|:-------|-----:|-----------:|-------:|-----------------:|
|PSTR    |    51|          26|      25|              51.0|
|CNAT    |    71|          33|      38|              46.5|
|SSID    |    63|          27|      36|              42.9|
|MCAV    |    67|          20|      47|              29.9|
|ORBI    |    57|           8|      49|              14.0|
|PAST    |    39|           4|      35|              10.3|
|MMEA    |    24|           1|      23|               4.2|

In [5]:
#count non-healthy coral conditions per location
non_healthy_counts_transects <- long_data %>%
  filter(health_status != "healthy") %>%                 
  count(Transect_num, sort = TRUE) %>%                   
  rename(non_healthy = n)                                

#total species counts per location
total_counts_transect <- long_data %>%
  count(Transect_num, sort = TRUE) %>%                   
  rename(total = n)                                      

#calculating percent unhealthy per location
coral_summary_transects <- total_counts_transect %>%
  left_join(non_healthy_counts_transects, by = "Transect_num") %>%
  mutate(
    non_healthy = replace_na(non_healthy, 0),              
    healthy = total - non_healthy,                        
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(Transect_num)                    

kable(coral_summary_transects, format = "markdown")         



| Transect_num| total| non_healthy| healthy| percent_unhealthy|
|------------:|-----:|-----------:|-------:|-----------------:|
|            1|    81|          35|      46|              43.2|
|            2|    87|          28|      59|              32.2|
|            3|   108|          42|      66|              38.9|
|            4|    96|          14|      82|              14.6|

In [6]:
#count non-healthy coral conditions per timepoint
non_healthy_by_time <- long_data %>%
  filter(health_status != "healthy") %>%                
  count(time_date, sort = TRUE) %>%                      
  rename(non_healthy = n) %>%                           
  arrange(time_date, desc(non_healthy))             

#total coral conditions per timepoint
total_counts_by_time <- long_data %>%
  count(time_date, sort = TRUE) %>%                     
  rename(total = n) %>%                                 
  arrange(time_date, desc(total))                   

#calculate percent unhealthy per species per timepoint
coral_summary_timepoint <- total_counts_by_time %>%
  left_join(non_healthy_by_time, by = c("time_date")) %>%
  mutate(
    non_healthy = replace_na(non_healthy, 0),                
    healthy = total - non_healthy,                           
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(desc(time_date))                             

# unhealthy species per transect & timepoint
non_healthy_transect_time <- long_data %>%
  filter(health_status != "healthy") %>%                        
  count(Transect_num, time_date, Species, sort = TRUE) %>%        
  rename(non_healthy = n) %>%                           
  arrange(Transect_num, time_date, desc(non_healthy))

kable(coral_summary_timepoint, format = "markdown")         



|time_date  | total| non_healthy| healthy| percent_unhealthy|
|:----------|-----:|-----------:|-------:|-----------------:|
|0010-01-20 |   142|          57|      85|              40.1|
|0009-01-20 |    88|           8|      80|               9.1|
|0007-01-20 |   142|          54|      88|              38.0|

In [7]:
#count non-healthy coral conditions per timepoint
non_healthy_by_time <- long_data %>%
  filter(health_status != "healthy") %>%                
  count(time_date, sort = TRUE) %>%                     
  rename(non_healthy = n) %>%                         
  arrange(time_date, desc(non_healthy))             

#total coral conditions per timepoint
total_counts_by_time <- long_data %>%
  count(time_date, sort = TRUE) %>%                    
  rename(total = n) %>%                               
  arrange(time_date, desc(total))                   

#calculate percent unhealthy per species per timepoint
coral_summary_timepoint <- total_counts_by_time %>%
  left_join(non_healthy_by_time, by = c("time_date")) %>%
  #add calculated columns: non_healthy, healthy, and percent_unhealthy
  mutate(
    non_healthy = replace_na(non_healthy, 0),                
    healthy = total - non_healthy,                          
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(desc(time_date))                               

# unhealthy species per transect & timepoint
non_healthy_transect_time <- long_data %>%
  filter(health_status != "healthy") %>%                           
  count(Transect_num, time_date, Species, sort = TRUE) %>%        
  rename(non_healthy = n) %>%                            
  arrange(Transect_num, time_date, desc(non_healthy))

#total coral conditions per species, timepoint, & transect
total_counts_by_transect_time <- long_data %>%
  count(Transect_num, time_date, Species, sort = TRUE) %>%
  rename(total = n) %>%
  arrange(time_date, desc(total)) 

#calculate percent unhealthy per species, timepoint, & transect
coral_summary_transect_time <- total_counts_by_transect_time %>%
  left_join(non_healthy_transect_time, by = c("time_date", "Species", "Transect_num")) %>%
  mutate(
    non_healthy = replace_na(non_healthy, 0),            
    healthy = total - non_healthy,                        
    percent_unhealthy = round((non_healthy / total) * 100, 1) ) %>%
  arrange(desc(time_date), Transect_num)                     

# creating obsidian friendly markdown
kable(coral_summary_transect_time, format = "markdown")    



| Transect_num|time_date  |Species | total| non_healthy| healthy| percent_unhealthy|
|------------:|:----------|:-------|-----:|-----------:|-------:|-----------------:|
|            1|0010-01-20 |SSID    |     7|           7|       0|             100.0|
|            1|0010-01-20 |CNAT    |     6|           6|       0|             100.0|
|            1|0010-01-20 |MCAV    |     5|           4|       1|              80.0|
|            1|0010-01-20 |ORBI    |     5|           1|       4|              20.0|
|            1|0010-01-20 |PAST    |     4|           1|       3|              25.0|
|            2|0010-01-20 |MCAV    |     7|           2|       5|              28.6|
|            2|0010-01-20 |CNAT    |     6|           6|       0|             100.0|
|            2|0010-01-20 |ORBI    |     6|           1|       5|              16.7|
|            2|0010-01-20 |PAST    |     5|           0|       5|               0.0|
|            2|0010-01-20 |SSID    |     5|           4|       

In [8]:
#now looking at sample data 
pan_samples <- read.csv("PAN-BDT_samples.csv")

#create a smaller dataframe with only selected columns
pan_samples_subset <- pan_samples %>%
  select(Month_year, Transect_num, Current_tag_num, Species, Sample_type, Health_status, Tubelabel_species, Sample_physical_location) %>%
  filter(
    Transect_num %in% c("1", "2", "3", "4"),            
    Sample_type %in% c("Core_RNAlater", "Core_EtOH"),    
    Sample_physical_location != "TX STATE") %>%         
  mutate(
    Species = str_trim(str_to_upper(Species)),                   
    Transect_num = str_trim(Transect_num),                       
    Health_status = str_trim(str_to_lower(Health_status))) %>%
  mutate(
    Species = case_when(
      Species %in% c("OANN", "OFAV") ~ "ORBI",TRUE ~ Species ))                     


In [12]:
# Determining shared species
species_transect_counts <- pan_samples_subset %>%
  distinct(Transect_num, Species) %>%                     #prevents duplicate combos 
  group_by(Species) %>%                  
  summarise(n_transects = n(), .groups = "drop")

kable(species_transect_counts, format = "markdown") 



|Species | n_transects|
|:-------|-----------:|
|CNAT    |           4|
|MCAV    |           4|
|MMEA    |           2|
|ORBI    |           4|
|PAST    |           3|
|PSTR    |           2|
|SSID    |           4|

In [11]:
# Filter for target species
shared_species <- species_transect_counts %>%
  filter(Species %in% c("CNAT", "PSTR", "MCAV", "SSID", "ORBI")) %>%
  pull(Species)

#filtering data by shared species 
T1toT4_samples_shared <- pan_samples_subset %>%
  filter(Species %in% shared_species)
nrow (T1toT4_samples_shared)

[1] 551

In [13]:
#Count non-healthy coral conditions per species
non_healthy_counts_species <- T1toT4_samples_shared %>%
  filter(Health_status != "healthy") %>%              
  count(Species, sort = TRUE) %>%                     
  rename(non_healthy = n)                         

# total species counts 
total_counts_species <- T1toT4_samples_shared %>%
  count(Species, sort = TRUE) %>%                    
  rename(total = n)                          

# calculating percent unhealthy per species
coral_summary_species <- total_counts_species %>%
  left_join(non_healthy_counts_species, by = "Species") %>%
  mutate(
    non_healthy = replace_na(non_healthy, 0),                
    healthy = total - non_healthy,                           
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(desc(percent_unhealthy))    

kable(coral_summary_species, format = "markdown")      



|Species | total| non_healthy| healthy| percent_unhealthy|
|:-------|-----:|-----------:|-------:|-----------------:|
|CNAT    |   129|          42|      87|              32.6|
|PSTR    |    92|          28|      64|              30.4|
|SSID    |   111|          27|      84|              24.3|
|MCAV    |   118|          22|      96|              18.6|
|ORBI    |   101|           9|      92|               8.9|

In [14]:
# Count non-healthy coral conditions per location
non_healthy_counts_transects <- T1toT4_samples_shared %>%
  filter(Health_status != "healthy") %>%                 #filter non-healthy
  count(Transect_num, sort = TRUE) %>%                   #count by transect
  rename(non_healthy = n)                                #rename the default n column

#total species counts per location
total_counts_transect <- T1toT4_samples_shared %>%
  count(Transect_num, sort = TRUE) %>%                   #count by transect
  rename(total = n)                                      #rename the default n column

#calculating percent unhealthy per location
coral_summary_transects <- total_counts_transect %>%
  left_join(non_healthy_counts_transects, by = "Transect_num") %>%
  # Add calculated columns: non_healthy, healthy, and percent_unhealthy
  mutate(
    non_healthy = replace_na(non_healthy, 0),                 #replace NA with 0
    healthy = total - non_healthy,                            #creating healthy column 
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(Transect_num)                     # highest percent unhealthy come first

kable(coral_summary_transects, format = "markdown")  



|Transect_num | total| non_healthy| healthy| percent_unhealthy|
|:------------|-----:|-----------:|-------:|-----------------:|
|1            |   122|          35|      87|              28.7|
|2            |   134|          30|     104|              22.4|
|3            |   174|          50|     124|              28.7|
|4            |   121|          13|     108|              10.7|

In [15]:
#count non-healthy coral conditions per timepoint
non_healthy_by_time <- T1toT4_samples_shared %>%
  filter(Health_status != "healthy") %>%                  #filter non-healthy
  count(Month_year, sort = TRUE) %>%                      #count by time point
  rename(non_healthy = n) %>%                             #rename the default n column
  arrange(Month_year, desc(non_healthy))             

# total coral conditions per timepoint
total_counts_by_time <- T1toT4_samples_shared %>%
  count(Month_year, sort = TRUE) %>%                      #count by Month_year
  rename(total = n) %>%                                   #rename the default n column
  arrange(Month_year, desc(total))                   

#calculate percent unhealthy per species per timepoint
coral_summary_timepoint <- total_counts_by_time %>%
  left_join(non_healthy_by_time, by = c("Month_year")) %>%
  #add calculated columns: non_healthy, healthy, and percent_unhealthy
  mutate(
    non_healthy = replace_na(non_healthy, 0),                 #replaces NA with 0
    healthy = total - non_healthy,                            #creating healthy column 
    percent_unhealthy = round((non_healthy / total) * 100, 1)) %>%
  arrange(desc(Month_year))                              #most recent date comes first

kable(coral_summary_timepoint, format = "markdown") 



| Month_year| total| non_healthy| healthy| percent_unhealthy|
|----------:|-----:|-----------:|-------:|-----------------:|
|     102023|   116|          46|      70|              39.7|
|      92022|   150|           2|     148|               1.3|
|      72024|   285|          80|     205|              28.1|